In [73]:
# -----------------------------
# 0) System Prompts
# -----------------------------

ROUTER_SYSTEM = """You are a research router. 
Decide if the topic needs web research. 
- needs_research: true if the topic is news, recent events (2024-2026), or specific facts.
- mode: 'open_book' if research is needed, 'closed_book' if it's general knowledge.
- queries: List of 3-5 search queries to find the latest info."""

RESEARCH_SYSTEM = """You are a research synthesizer. 
Convert raw search results into a clean list of evidence items with titles and URLs."""

ORCH_SYSTEM = """You are a technical blog architect.
Output ONLY a JSON object. Keep it short and simple.
Example format:
{
  "blog_title": "Title here",
  "audience": "Developers",
  "tasks": [
    {
      "id": 1, 
      "title": "Introduction", 
      "goal": "Explain the basics", 
      "bullets": ["Point 1", "Point 2", "Point 3"],
      "target_words": 200
    }
  ]
}"""


In [74]:
!pip install langchain-community langchain-tavily


In [75]:
import os
import operator
import json
import ast
import re
from pathlib import Path
from typing import TypedDict, List, Optional, Literal, Annotated

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.types import Send

from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults

# Load your tokens from .env
load_dotenv()


True

In [76]:
# -----------------------------
# 1) Schemas (Relaxed)
# -----------------------------
class Task(BaseModel):
    id: int
    title: str

    # We use defaults (= "") so the code doesn't crash if the AI misses a field
    goal: str = ""
    bullets: List[str] = []
    target_words: int = 300 

    tags: List[str] = Field(default_factory=list)
    requires_research: bool = False
    requires_citations: bool = False
    requires_code: bool = False


class Plan(BaseModel):
    blog_title: str
    audience: str = "Developers"
    tone: str = "Technical"
    blog_kind: str = "explainer"
    constraints: List[str] = Field(default_factory=list)
    tasks: List[Task] = []


class EvidenceItem(BaseModel):
    title: str
    url: str
    published_at: Optional[str] = None
    snippet: Optional[str] = None
    source: Optional[str] = None


class RouterDecision(BaseModel):
    needs_research: bool
    mode: str = "closed_book"
    queries: List[str] = Field(default_factory=list)


class EvidencePack(BaseModel):
    evidence: List[EvidenceItem] = Field(default_factory=list)


In [77]:
class State(TypedDict):
    topic: str

    # routing / research
    mode: str
    needs_research: bool
    queries: List[str]
    evidence: List[EvidenceItem]
    plan: Optional[Plan]

    # workers
    sections: Annotated[List[tuple[int, str]], operator.add]  # (task_id, section_md)
    final: str

In [78]:
# -----------------------------
# 2) LLM Initialization
# -----------------------------
llm_endpoint = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    task="text-generation",
    max_new_tokens=2048,
    do_sample=False,
)
llm = ChatHuggingFace(llm=llm_endpoint)


In [79]:
def router_node(state: State) -> dict:
    topic = state["topic"]
    prompt = (
        f"{ROUTER_SYSTEM}\n\n"
        f"TOPIC: {topic}\n\n"
        "Output ONLY a JSON object: {\"needs_research\": bool, \"mode\": \"string\", \"queries\": [\"query1\", \"query2\"]}"
    )
    
    response = llm.invoke([HumanMessage(content=prompt)])
    content = response.content.strip()
    
    # Extract JSON between curly braces
    match = re.search(r'\{.*\}', content, re.DOTALL)
    if not match:
        return {"needs_research": False, "mode": "closed_book", "queries": []}
    
    try:
        data = json.loads(match.group())
        return {
            "needs_research": data.get("needs_research", False),
            "mode": data.get("mode", "closed_book"),
            "queries": data.get("queries", []),
        }
    except:
        return {"needs_research": False, "mode": "closed_book", "queries": []}


In [80]:
def research_node(state: State) -> dict:
    queries = (state.get("queries", []) or [])
    max_results = 6
    raw_results = []

    for q in queries:
        raw_results.extend(_tavily_search(q, max_results=max_results))

    if not raw_results:
        return {"evidence": []}

    prompt = (
        f"{RESEARCH_SYSTEM}\n\n"
        f"Raw results:\n{raw_results}\n\n"
        "Output ONLY a JSON object: {\"evidence\": [{\"title\": \"...\", \"url\": \"...\", \"snippet\": \"...\"}]}"
    )
    
    response = llm.invoke([HumanMessage(content=prompt)])
    content = response.content.strip()
    
    match = re.search(r'\{.*\}', content, re.DOTALL)
    if not match:
        return {"evidence": []}
        
    try:
        try:
            data = json.loads(match.group())
        except:
            data = ast.literal_eval(match.group())
        
        # Deduplicate by URL
        dedup = {}
        for e_dict in data.get("evidence", []):
            url = e_dict.get("url")
            if url and url not in dedup:
                dedup[url] = EvidenceItem(**e_dict)
        
        return {"evidence": list(dedup.values())}
    except Exception as e:
        print(f"❌ Research synthesis failed: {e}")
        return {"evidence": []}


In [81]:
def orchestrator_node(state: State) -> dict:
    evidence = state.get("evidence", [])
    mode = state.get("mode", "closed_book")
    
    prompt = (
        f"{ORCH_SYSTEM}\n\n"
        f"Topic: {state['topic']}\n"
        f"Mode: {mode}\n"
        f"Evidence: {[e.title for e in evidence][:5]}\n\n"
        "Output ONLY a valid JSON object matching the Plan schema."
    )

    response = llm.invoke([HumanMessage(content=prompt)])
    content = response.content.strip()
    
    match = re.search(r'\{.*\}', content, re.DOTALL)
    if not match:
        return {"plan": Plan(blog_title="Error Plan", tasks=[])}
        
    try:
        try:
            plan_dict = json.loads(match.group())
        except:
            plan_dict = ast.literal_eval(match.group())
            
        plan = Plan(**plan_dict)
        return {"plan": plan}
    except Exception as e:
        print(f"❌ Planning failed: {e}")
        # Simple fallback so the graph doesn't stop
        return {"plan": Plan(blog_title=state['topic'], tasks=[Task(id=1, title="Intro", goal="Start here", bullets=["B1", "B2"])])}


In [82]:
# -----------------------------
# 6) Fanout
# -----------------------------
def fanout(state: State):
    return [
        Send(
            "worker",
            {
                "task": task.model_dump(),
                "topic": state["topic"],
                "mode": state["mode"],
                "plan": state["plan"].model_dump(),
                "evidence": [e.model_dump() for e in state.get("evidence", [])],
            },
        )
        for task in state["plan"].tasks
    ]



In [83]:
# -----------------------------
# 7) Worker (write one section)
# -----------------------------
WORKER_SYSTEM = """You are a senior technical writer and developer advocate.
Write ONE section of a technical blog post in Markdown.

Hard constraints:
- Follow the provided Goal and cover ALL Bullets in order (do not skip or merge bullets).
- Stay close to Target words (±15%).
- Output ONLY the section content in Markdown (no blog title H1, no extra commentary).
- Start with a '## <Section Title>' heading.

Scope guard:
- If blog_kind == "news_roundup": do NOT turn this into a tutorial/how-to guide.
  Do NOT teach web scraping, RSS, automation, or "how to fetch news" unless bullets explicitly ask for it.
  Focus on summarizing events and implications.

Grounding policy:
- If mode == open_book:
  - Do NOT introduce any specific event/company/model/funding/policy claim unless it is supported by provided Evidence URLs.
  - For each event claim, attach a source as a Markdown link: ([Source](URL)).
  - Only use URLs provided in Evidence. If not supported, write: "Not found in provided sources."
- If requires_citations == true:
  - For outside-world claims, cite Evidence URLs the same way.
- Evergreen reasoning is OK without citations unless requires_citations is true.

Code:
- If requires_code == true, include at least one minimal, correct code snippet relevant to the bullets.

Style:
- Short paragraphs, bullets where helpful, code fences for code.
- Avoid fluff/marketing. Be precise and implementation-oriented.
"""

def worker_node(payload: dict) -> dict:
    
    task = Task(**payload["task"])
    plan = Plan(**payload["plan"])
    evidence = [EvidenceItem(**e) for e in payload.get("evidence", [])]
    topic = payload["topic"]
    mode = payload.get("mode", "closed_book")

    bullets_text = "\n- " + "\n- ".join(task.bullets)

    evidence_text = ""
    if evidence:
        evidence_text = "\n".join(
            f"- {e.title} | {e.url} | {e.published_at or 'date:unknown'}".strip()
            for e in evidence[:20]
        )

    section_md = llm.invoke(
        [
            SystemMessage(content=WORKER_SYSTEM),
            HumanMessage(
                content=(
                    f"Blog title: {plan.blog_title}\n"
                    f"Audience: {plan.audience}\n"
                    f"Tone: {plan.tone}\n"
                    f"Blog kind: {plan.blog_kind}\n"
                    f"Constraints: {plan.constraints}\n"
                    f"Topic: {topic}\n"
                    f"Mode: {mode}\n\n"
                    f"Section title: {task.title}\n"
                    f"Goal: {task.goal}\n"
                    f"Target words: {task.target_words}\n"
                    f"Tags: {task.tags}\n"
                    f"requires_research: {task.requires_research}\n"
                    f"requires_citations: {task.requires_citations}\n"
                    f"requires_code: {task.requires_code}\n"
                    f"Bullets:{bullets_text}\n\n"
                    f"Evidence (ONLY use these URLs when citing):\n{evidence_text}\n"
                )
            ),
        ]
    ).content.strip()

    return {"sections": [(task.id, section_md)]}


In [84]:
# -----------------------------
# 8) Reducer (merge + save)
# -----------------------------
def reducer_node(state: State) -> dict:

    plan = state["plan"]

    ordered_sections = [md for _, md in sorted(state["sections"], key=lambda x: x[0])]
    body = "\n\n".join(ordered_sections).strip()
    final_md = f"# {plan.blog_title}\n\n{body}\n"

    filename = f"{plan.blog_title}.md"
    Path(filename).write_text(final_md, encoding="utf-8")

    return {"final": final_md}


In [85]:
def route_next(state: State) -> str:
    return "research" if state["needs_research"] else "orchestrator"


In [86]:
# -----------------------------
# 9) Build the Research Graph
# -----------------------------
g = StateGraph(State)

# Add our nodes
g.add_node("router", router_node)
g.add_node("research", research_node)
g.add_node("orchestrator", orchestrator_node)
g.add_node("worker", worker_node)
g.add_node("reducer", reducer_node)

# Define the flow
g.add_edge(START, "router")

# The Router decides: Go to research or go straight to planning?
g.add_conditional_edges(
    "router", 
    route_next, 
    {"research": "research", "orchestrator": "orchestrator"}
)

g.add_edge("research", "orchestrator")

# Orchestrator sends tasks to workers
g.add_conditional_edges("orchestrator", fanout, ["worker"])

# Workers go to reducer
g.add_edge("worker", "reducer")
g.add_edge("reducer", END)

# Compile the graph into 'app'
app = g.compile()

print("✅ Graph compiled successfully! 'app' is now ready.")


✅ Graph compiled successfully! 'app' is now ready.


In [87]:
# -----------------------------
# 10) Runner
# -----------------------------
def run(topic: str):
    out = app.invoke(
        {
            "topic": topic,
            "mode": "",
            "needs_research": False,
            "queries": [],
            "evidence": [],
            "plan": None,
            "sections": [],
            "final": "",
        }
    )

    return out


In [88]:
#run("Write a blog on Open Source LLMs in 2026")
run("State of Multimodal LLMs in 2026")

{'topic': 'State of Multimodal LLMs in 2026',
 'mode': 'closed_book',
 'needs_research': False,
 'queries': [],
 'evidence': [],
 'plan': Plan(blog_title='Unleashing the Power of State of Multimodal LLMs in 2026', audience='Researchers', tone='Technical', blog_kind='explainer', constraints=[], tasks=[Task(id=1, title='Introduction', goal='Explain multimodal LLMs and their potential', bullets=['Cover history', 'Introduce key research achievements', 'Explain how multimodal LLMs differ from text-only models', 'Highlight some successful applications'], target_words=1500, tags=[], requires_research=False, requires_citations=False, requires_code=False), Task(id=2, title='Architecture', goal='Describe the state-of-the-art architectures', bullets=['AutoEncoders', 'VLP', 'OCR', 'Multisense', 'OCR', 'Alignment'], target_words=1000, tags=[], requires_research=False, requires_citations=False, requires_code=False), Task(id=3, title='Future directions', goal='Discuss strengths and limitations', bull